# Deepfake Audio Detection


To improve speed:
- Cached mel spectrograms (compute once, reuse every epoch)
- num_workers=2 with multiprocessing
- Mixed precision training (AMP) - 2x faster on T4
- Lightweight CNN with channel attention


In [ ]:
import os, glob, random, json
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                              roc_curve, balanced_accuracy_score)
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


## 1. Dataset

In [ ]:
BASE      = '/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm'
TRAIN_DIR = os.path.join(BASE, 'training')
VAL_DIR   = os.path.join(BASE, 'validation')
TEST_DIR  = os.path.join(BASE, 'testing')

def collect_files(data_dir):
    rows = []
    for sub in os.listdir(data_dir):
        sub_path = os.path.join(data_dir, sub)
        if not os.path.isdir(sub_path): continue
        sub_l = sub.lower()
        if sub_l == 'real':   label = 0
        elif sub_l == 'fake': label = 1
        else: continue
        for f in glob.glob(os.path.join(sub_path, '*.wav')):
            rows.append((f, label))
    return pd.DataFrame(rows, columns=['path', 'label'])

train_df = collect_files(TRAIN_DIR)
val_df   = collect_files(VAL_DIR)
test_df  = collect_files(TEST_DIR)

print('Train:', train_df.shape, train_df['label'].value_counts().to_dict())
print('Val:',   val_df.shape,   val_df['label'].value_counts().to_dict())
print('Test:',  test_df.shape,  test_df['label'].value_counts().to_dict())
assert len(train_df) > 0, 'No files found'

# --- Diagnostic: class imbalance ratio per split (drives §2 below) ---
print()
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    vc = df['label'].value_counts().to_dict()
    n0, n1 = vc.get(0, 0), vc.get(1, 0)
    ratio = max(n0, n1) / max(1, min(n0, n1))
    majority = 'Genuine' if n0 >= n1 else 'Deepfake'
    print(f'{name:5s} -> Genuine={n0:6d}  Deepfake={n1:6d}  '
          f'imbalance={ratio:.2f}x  majority={majority}')


## 2. Balance the training data (capped oversampling + data-driven class weights)



In [ ]:
genuine_df  = train_df[train_df['label'] == 0]
deepfake_df = train_df[train_df['label'] == 1]
n_genuine, n_deepfake = len(genuine_df), len(deepfake_df)

MAX_OVERSAMPLE_RATIO = 3.0  # cap on how many times a minority clip can be repeated

if n_genuine >= n_deepfake:
    majority_df, minority_df = genuine_df, deepfake_df
else:
    majority_df, minority_df = deepfake_df, genuine_df

target_minority = min(len(majority_df), int(len(minority_df) * MAX_OVERSAMPLE_RATIO))
minority_oversampled = minority_df.sample(n=target_minority, replace=True, random_state=SEED)

train_df_balanced = (pd.concat([majority_df, minority_oversampled])
                       .sample(frac=1, random_state=SEED)
                       .reset_index(drop=True))

counts = train_df_balanced['label'].value_counts().to_dict()
print(f'Balanced training set: {counts}')
print(f'Total: {len(train_df_balanced)} samples '
      f'(oversample ratio used: {target_minority / max(1, len(minority_df)):.2f}x)')

# Data-driven class weights for any *residual* imbalance (fed to FocalLoss in §5)
n0 = counts.get(0, 1)
n1 = counts.get(1, 1)
class_weights = torch.tensor([
    (n0 + n1) / (2.0 * n0),
    (n0 + n1) / (2.0 * n1),
], dtype=torch.float32)
print('Class weights [Genuine, Deepfake]:', [round(w, 3) for w in class_weights.tolist()])


## 3. Pre-cache ALL mel spectrograms into RAM
Compute each mel spectrogram once and store in memory. Every epoch reuses cached data instead of reloading audio files from disk.

In [ ]:
SR        = 16000
DURATION  = 4.0
N_SAMPLES = int(SR * DURATION)
N_MELS    = 128

def load_audio(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    else:
        y = y[:N_SAMPLES]
    return y

def to_logmel(y):
    mel    = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, n_fft=1024, hop_length=256)
    logmel = librosa.power_to_db(mel, ref=np.max)
    logmel = (logmel - logmel.min()) / (logmel.max() - logmel.min() + 1e-6)
    return logmel.astype(np.float32)

def cache_mels(df, desc='Caching'):
    mels = []
    for path in tqdm(df['path'].values, desc=desc):
        try:
            y   = load_audio(path)
            mel = to_logmel(y)
        except Exception:
            mel = np.zeros((N_MELS, int(N_SAMPLES/256)+1), dtype=np.float32)
        mels.append(mel)
    return mels

print('Caching training mels...')
train_mels = cache_mels(train_df_balanced, 'Train')
print('Caching validation mels...')
val_mels   = cache_mels(val_df,            'Val')
print('Caching test mels...')
test_mels  = cache_mels(test_df,           'Test')
print('All mels cached!')


## 4. uses cached mels

In [ ]:
def spec_augment(mel, freq_mask=20, time_mask=30, n_freq_masks=2, n_time_masks=2):
    mel = mel.copy()
    n_mels, n_time = mel.shape
    for _ in range(n_freq_masks):
        if n_mels > freq_mask:
            f0 = np.random.randint(0, n_mels - freq_mask)
            mel[f0:f0+freq_mask, :] = 0
    for _ in range(n_time_masks):
        if n_time > time_mask:
            t0 = np.random.randint(0, n_time - time_mask)
            mel[:, t0:t0+time_mask] = 0
    return mel

class CachedDataset(Dataset):
    def __init__(self, mels, labels, augment=False):
        self.mels    = mels
        self.labels  = labels
        self.augment = augment

    def __len__(self):
        return len(self.mels)

    def __getitem__(self, idx):
        mel = self.mels[idx].copy()
        if self.augment:
            mel = spec_augment(mel)
            if np.random.rand() < 0.4:
                mel = np.clip(mel + np.random.randn(*mel.shape).astype(np.float32) * 0.02, 0, 1)
        return torch.from_numpy(mel).unsqueeze(0), torch.tensor(self.labels[idx], dtype=torch.long)

train_ds = CachedDataset(train_mels, train_df_balanced['label'].values, augment=True)
val_ds   = CachedDataset(val_mels,   val_df['label'].values,            augment=False)
test_ds  = CachedDataset(test_mels,  test_df['label'].values,           augment=False)

BATCH_SIZE   = 64   # larger batch = faster training
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('Mel shape:', train_ds[0][0].shape)
print('Train batches:', len(train_loader))


## 5. Focal Loss (data-driven class weights, not a hard-coded alpha)



In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, inputs, targets):
        # Unweighted CE -> used purely to get pt = exp(-ce) for the focal term
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_term = (1 - pt) ** self.gamma

        if self.class_weights is not None:
            alpha_t = self.class_weights.to(inputs.device)[targets]
        else:
            alpha_t = 1.0

        return (alpha_t * focal_term * ce_loss).mean()


## 6. Model: Lightweight AudioCNN with Attention

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x).unsqueeze(-1).unsqueeze(-1)

class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.attn = ChannelAttention(channels)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(x + self.attn(self.block(x)))

class AudioCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            # Stem
            nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            # Block 1
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            ResBlock(64), nn.MaxPool2d(2), nn.Dropout2d(0.1),
            # Block 2
            nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            ResBlock(128), nn.MaxPool2d(2), nn.Dropout2d(0.2),
            # Block 3
            nn.Conv2d(128, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = AudioCNN().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'AudioCNN | Parameters: {total_params:,}')


## 7. Training with Mixed Precision (AMP)



In [ ]:
EPOCHS       = 18
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 7
GAMMA        = 2.0

criterion = FocalLoss(class_weights=class_weights, gamma=GAMMA)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
scaler    = GradScaler()   # mixed precision scaler

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.set_grad_enabled(train):
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train:
                optimizer.zero_grad()
                with autocast():          # mixed precision forward pass
                    out  = model(x)
                    loss = criterion(out, y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                with autocast():
                    out  = model(x)
                    loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            all_preds.extend(out.argmax(1).detach().cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_preds), all_labels, all_preds

best_val_bal_acc  = -1.0
best_val_loss     = float('inf')
best_val_acc      = 0.0
epochs_no_improve = 0
history           = []

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc, _, _                          = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_labels_ep, val_preds_ep = run_epoch(val_loader,   train=False)
    val_bal_acc = balanced_accuracy_score(val_labels_ep, val_preds_ep)
    scheduler.step(val_loss)
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'train_acc': tr_acc,
                    'val_loss': val_loss, 'val_acc': val_acc, 'val_bal_acc': val_bal_acc})
    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'train_loss={tr_loss:.4f} acc={tr_acc:.4f} | '
          f'val_loss={val_loss:.4f} acc={val_acc:.4f} bal_acc={val_bal_acc:.4f} | '
          f'lr={optimizer.param_groups[0]["lr"]:.2e}')

    if val_bal_acc > best_val_bal_acc + 1e-4:
        best_val_bal_acc = val_bal_acc
        best_val_loss    = val_loss
        best_val_acc     = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print('  -> best model saved (highest balanced accuracy so far)')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print('  -> early stopping triggered')
            break

print(f'Done | Best val_bal_acc={best_val_bal_acc:.4f} '
      f'val_loss={best_val_loss:.4f} val_acc={best_val_acc:.4f}')


In [ ]:
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='train')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('epoch')
axes[1].plot(hist_df['epoch'], hist_df['train_acc'], label='train')
axes[1].plot(hist_df['epoch'], hist_df['val_acc'],   label='val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('epoch')
axes[2].plot(hist_df['epoch'], hist_df['val_bal_acc'], label='val balanced acc', color='green')
axes[2].axhline(0.75, color='red', linestyle='--', label='75% per-class target')
axes[2].set_title('Validation Balanced Accuracy'); axes[2].legend(); axes[2].set_xlabel('epoch')
plt.savefig('training_curves.png', bbox_inches='tight')
plt.show()


## 8. Probability calibration (temperature scaling)



In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

def get_logits(loader):
    """Run the model over a loader and return (logits, true_labels)."""
    logits_list, labels_list = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, leave=False):
            with autocast():
                out = model(x.to(DEVICE))
            logits_list.append(out.float().cpu())
            labels_list.append(y)
    return torch.cat(logits_list), torch.cat(labels_list)

val_logits, val_labels_t = get_logits(val_loader)

def val_nll(temperature):
    return F.cross_entropy(val_logits / temperature, val_labels_t).item()

uncalibrated_nll = val_nll(1.0)
best_T, best_T_nll = 1.0, uncalibrated_nll
for T in np.arange(0.25, 5.0, 0.05):
    n = val_nll(T)
    if n < best_T_nll:
        best_T_nll, best_T = n, T

print(f'Validation NLL @ T=1.00 (uncalibrated): {uncalibrated_nll:.4f}')
print(f'Validation NLL @ T={best_T:.2f} (calibrated)  : {best_T_nll:.4f}')

val_probs = torch.softmax(val_logits / best_T, dim=1)[:, 1].numpy()
val_true  = val_labels_t.numpy()

print('\n--- P(deepfake) distribution on VALIDATION set (per true class) ---')
for cls, name in [(0, 'Genuine'), (1, 'Deepfake')]:
    p = val_probs[val_true == cls]
    print(f'true={name:8s} -> min={p.min():.3f}  mean={p.mean():.3f}  '
          f'median={np.median(p):.3f}  max={p.max():.3f}  (n={len(p)})')


## 9. Fine-grained threshold search (on calibrated probabilities)



In [ ]:
results = []
for thresh in np.arange(0.01, 0.99, 0.001):
    preds   = (val_probs >= thresh).astype(int)
    cm_val  = confusion_matrix(val_true, preds, labels=[0, 1])
    per_cls = cm_val.diagonal() / cm_val.sum(axis=1).clip(min=1)
    results.append({
        'thresh':       thresh,
        'acc':          accuracy_score(val_true, preds),
        'f1':           f1_score(val_true, preds, zero_division=0),
        'bal_acc':      per_cls.mean(),
        'genuine_acc':  per_cls[0],
        'deepfake_acc': per_cls[1],
    })
results_df = pd.DataFrame(results)

REQ = dict(acc=0.80, f1=0.80, genuine_acc=0.75, deepfake_acc=0.75)
valid = results_df[
    (results_df['acc']          >= REQ['acc']) &
    (results_df['f1']           >= REQ['f1']) &
    (results_df['genuine_acc']  >= REQ['genuine_acc']) &
    (results_df['deepfake_acc'] >= REQ['deepfake_acc'])
]

if len(valid) > 0:
    best_row = valid.loc[valid['bal_acc'].idxmax()]
    print(f'{len(valid)} threshold(s) on validation satisfy ALL requirements - '
          f'picking the one with the highest balanced accuracy.')
else:
    best_row = results_df.loc[results_df['bal_acc'].idxmax()]
    print('No single threshold satisfies every requirement on validation - '
          'falling back to the balanced-accuracy maximizer.')

best_thresh = float(best_row['thresh'])
print(best_row)
print(f'\nSelected threshold: {best_thresh:.3f}  |  Calibration T: {best_T:.2f}')


## 10. Final evaluation on test set

In [ ]:
test_logits, test_labels_t = get_logits(test_loader)
all_probs  = torch.softmax(test_logits / best_T, dim=1)[:, 1].numpy()
all_labels = test_labels_t.numpy()
all_preds  = (all_probs >= best_thresh).astype(int)

print('--- P(deepfake) distribution on TEST set (per true class) ---')
for cls, name in [(0, 'Genuine'), (1, 'Deepfake')]:
    p = all_probs[all_labels == cls]
    print(f'true={name:8s} -> min={p.min():.3f}  mean={p.mean():.3f}  '
          f'median={np.median(p):.3f}  max={p.max():.3f}  (n={len(p)})')

print('\nPrediction distribution:', np.bincount(all_preds, minlength=2))

acc           = accuracy_score(all_labels, all_preds)
f1            = f1_score(all_labels, all_preds)
cm            = confusion_matrix(all_labels, all_preds)
per_class_acc = cm.diagonal() / cm.sum(axis=1)

fpr, tpr, _ = roc_curve(all_labels, all_probs)
fnr         = 1 - tpr
eer_idx     = np.nanargmin(np.absolute(fnr - fpr))
eer         = (fpr[eer_idx] + fnr[eer_idx]) / 2

print(f'\nCalibration T  : {best_T:.2f}')
print(f'Threshold      : {best_thresh:.3f}')
print(f'Accuracy       : {acc:.4f}')
print(f'F1 Score       : {f1:.4f}')
print(f'EER            : {eer*100:.2f}%')
print(f'Confusion Matrix [0=Genuine, 1=Deepfake]:')
print(cm)
print(f'Per-class acc  : Genuine={per_class_acc[0]:.4f}, Deepfake={per_class_acc[1]:.4f}')

print('\n--- Requirement check ---')
print(f'Accuracy  >= 80% : {"PASS" if acc >= 0.80 else "FAIL"} ({acc*100:.2f}%)')
print(f'EER       <= 12% : {"PASS" if eer <= 0.12 else "FAIL"} ({eer*100:.2f}%)')
print(f'F1        >= 80% : {"PASS" if f1  >= 0.80 else "FAIL"} ({f1*100:.2f}%)')
print(f'Genuine   >= 75% : {"PASS" if per_class_acc[0] >= 0.75 else "FAIL"} ({per_class_acc[0]*100:.2f}%)')
print(f'Deepfake  >= 75% : {"PASS" if per_class_acc[1] >= 0.75 else "FAIL"} ({per_class_acc[1]*100:.2f}%)')


In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Genuine', 'Deepfake'],
            yticklabels=['Genuine', 'Deepfake'])
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()
